<a href="https://colab.research.google.com/github/vnsantos/projeto_mrp_costing/blob/main/MRP_Costing_CBASE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import io
from google.colab import files

# --- PROCESSAMENTO DOS ARQUIVOS ---

# Variável global para guardar a Cia identificada dinamicamente
companhia_detectada = "TAP"

# --- CLASSIFICAÇÃO PERSONALIZADA (COLUNA CLASS) ---
def obter_classificacao_classe(codigo):
    global companhia_detectada
    if pd.isna(codigo):
        return None
    codigo_str = str(codigo).strip().upper()

    # Pegas as letras iniciais significativas da cia (removendo números e espaços)
    cia_letras = re.sub(r'[^A-Z]', '', companhia_detectada.upper())

    if codigo_str.startswith(('R', 'Y')):
        return "R"
    elif codigo_str.startswith('13'):
        return "P"
    elif codigo_str.startswith(('T', 'C')):
        # Verifica se o caractere após o 'T' ou 'C' é um número
        if len(codigo_str) > 1 and codigo_str[1].isdigit():
            return None
        else:
            # Se iniciar com as letras da Cia (ex: TAP começa com T, AA começa com A/AM...)
            if cia_letras and codigo_str.startswith(cia_letras[:2]):
                return "CIA"
            return "CIA"
    else:
        # Verifica se inicia com as mesmas letras iniciais da companhia (ex: TP para TAP, AA para American Airlines)
        # Considera os 2 primeiros caracteres da Cia
        prefixo_cia = cia_letras[:2] if len(cia_letras) >= 2 else cia_letras
        if prefixo_cia and codigo_str.startswith(prefixo_cia):
            return "CIA"
        else:
            return "ESP"

def processar_meal_definition_unificado():
    global companhia_detectada
    print("--- Selecione o arquivo Meal Definition ---")
    uploaded = files.upload()
    if not uploaded: return pd.DataFrame(), set()
    arquivo = list(uploaded.keys())[0]

    # Lê o arquivo para capturar a primeira célula (A1) -> Linha 0, Coluna 0
    df_identificacao = pd.read_excel(io.BytesIO(uploaded[arquivo]), header=None, nrows=1)
    if not df_identificacao.empty and df_identificacao.shape[1] > 0:
        valor_a1 = str(df_identificacao.iloc[0, 0]).strip()
        if valor_a1 and valor_a1.lower() != 'nan':
            companhia_detectada = valor_a1
            print(f"Companhia identificada na célula A1: {companhia_detectada}")

    # Lê o arquivo Excel a partir da linha correta para a estrutura de dados
    df = pd.read_excel(io.BytesIO(uploaded[arquivo]), header=1)

    colunas_originais = list(df.columns)
    # Normaliza nomes de colunas: string, sem espaços nas pontas, tudo maiúsculo e sem espaços/subtrações internas se possível
    df.columns = df.columns.astype(str).str.strip().str.upper()
    df.columns = df.columns.str.replace(" ", "").str.replace("_", "").str.replace("-", "")

    mapa_renomear = {}
    if len(colunas_originais) > 0: mapa_renomear[colunas_originais[0].strip().upper().replace(" ", "").replace("_", "").replace("-", "")] = "Servico"
    if len(colunas_originais) > 3: mapa_renomear[colunas_originais[3].strip().upper().replace(" ", "").replace("_", "").replace("-", "")] = "Classe"
    if len(colunas_originais) > 6: mapa_renomear[colunas_originais[6].strip().upper().replace(" ", "").replace("_", "").replace("-", "")] = "Rotation"
    if len(colunas_originais) > 9: mapa_renomear[colunas_originais[9].strip().upper().replace(" ", "").replace("_", "").replace("-", "")] = "Cardapio_Base_ColJ"

    mapa_geral = {
        "CPACKINGLIST": "CPACKINGLIST",
        "PRODTEXT": "Descricao",
        "PROD_TEXT": "Descricao",
        "MODE": "Mode",
        "VALUE": "Value"
    }
    mapa_geral.update(mapa_renomear)
    df = df.rename(columns=mapa_geral)

    # Tentativa inteligente de encontrar a coluna de Packing List caso não tenha mapeado perfeitamente
    if "CPACKINGLIST" not in df.columns:
        for col in df.columns:
            if "PACK" in col or "CPACK" in col:
                df = df.rename(columns={col: "CPACKINGLIST"})
                break

    if "CPACKINGLIST" not in df.columns:
        print(f"\n[ERRO] Não foi possível encontrar a coluna 'CPACKINGLIST' no arquivo!")
        print(f"Colunas identificadas no seu arquivo: {list(df.columns)}")
        return pd.DataFrame(), set()

    df["Class_Meal"] = df["CPACKINGLIST"].apply(obter_classificacao_classe)

    colunas_necessarias = ["Servico", "Rotation", "CPACKINGLIST", "Descricao", "Mode", "Value", "Classe", "Cardapio_Base_ColJ", "Class_Meal"]
    cols_existentes = [c for c in colunas_necessarias if c in df.columns]

    # Obtém o conjunto de todos os CPACKINGLISTs do Meal Definition em maiúsculo
    set_cpacking = set(df["CPACKINGLIST"].dropna().astype(str).str.strip().str.upper().unique())

    return df[cols_existentes].copy(), set_cpacking

def processar_costing_sequencial(set_codigos_meal):
    print("--- Selecione o arquivo Costing Aberto---")
    uploaded = files.upload()
    if not uploaded: return {}, {}
    arquivo = list(uploaded.keys())[0]

    # Lê o arquivo Costing linha por linha
    df = pd.read_excel(io.BytesIO(uploaded[arquivo]), header=None)

    mapa_receitas = {}
    mapa_custos_pais = {}
    cpacking_atual = None

    for idx in range(len(df)):
        codigo = str(df.iloc[idx, 0]).strip()
        if codigo == "nan": continue

        codigo_upper = codigo.upper()

        if codigo_upper in set_codigos_meal:
            cpacking_atual = codigo_upper
            mapa_receitas[cpacking_atual] = []

            # Captura custo próprio do Pai
            qtgross_pai = df.iloc[idx, 6]
            un_val_pai = df.iloc[idx, 7]

            if pd.notnull(qtgross_pai) and str(qtgross_pai).strip() != "" and str(qtgross_pai).strip().lower() != "nan":
                custo_pai = qtgross_pai
            elif pd.notnull(un_val_pai) and str(un_val_pai).strip() != "" and str(un_val_pai).strip().lower() != "nan":
                custo_pai = un_val_pai
            else:
                custo_pai = 0.0

            mapa_custos_pais[cpacking_atual] = custo_pai
            continue

        if cpacking_atual is not None:
            classe_item_calculada = obter_classificacao_classe(codigo)

            ctext_short = df.iloc[idx, 3] if pd.notnull(df.iloc[idx, 3]) else None
            qtnet = df.iloc[idx, 4] if pd.notnull(df.iloc[idx, 4]) else None
            yield_val = df.iloc[idx, 5] if pd.notnull(df.iloc[idx, 5]) else None

            qtgross = df.iloc[idx, 6]
            un_val = df.iloc[idx, 7]

            if pd.notnull(qtgross) and str(qtgross).strip() != "" and str(qtgross).strip().lower() != "nan":
                custo_unitario_final = qtgross
            elif pd.notnull(un_val) and str(un_val).strip() != "" and str(un_val).strip().lower() != "nan":
                custo_unitario_final = un_val
            else:
                custo_unitario_final = 0.0

            mapa_receitas[cpacking_atual].append({
                "Codigo": codigo,
                "Descricao_Item": str(df.iloc[idx, 1]).strip(),
                "QTNET": qtnet,
                "YIELD": ctext_short,
                "UN_NOVO": yield_val,
                "CUSTO_FINAL": custo_unitario_final,
                "Class_Costing": classe_item_calculada
            })

    return mapa_receitas, mapa_custos_pais

# --- EXECUÇÃO DO PROCESSO ---

df_meal_unif, set_codigos_meal = processar_meal_definition_unificado()

if not df_meal_unif.empty:
    mapa_costing, mapa_custos_pais = processar_costing_sequencial(set_codigos_meal)

    if mapa_costing:
        resultado_unificado = []

        for row in df_meal_unif.to_dict("records"):
            cpacking_raiz = str(row.get("CPACKINGLIST", "")).strip().upper()
            descricao_original = row.get("Descricao")
            cardapio_col_j = row.get("Cardapio_Base_ColJ")

            # Registro pai na listagem (MealD)
            # Se o pai original possuir uma descrição válida (na Coluna J ou Descricao do Meal Definition),
            # nós mantemos esse nome intacto como Produto e receita.
            nome_produto_pai = cardapio_col_j if pd.notnull(cardapio_col_j) else descricao_original

            resultado_unificado.append([
                "GIG",
                "CBase",
                "01/07/2026",
                companhia_detectada,
                row.get("Servico"),
                None,                              # Preenchido via propagação
                "MealD",
                row.get("Class_Meal"),
                row.get("CPACKINGLIST"),
                nome_produto_pai,                  # Mantém o produto principal do código original intacto
                row.get("Classe"),
                row.get("Rotation"),
                None,
                None,
                "UN",
                mapa_custos_pais.get(cpacking_raiz, 0.0)
            ])

            # Traz todos os itens associados ao CPACKINGLIST sequencialmente
            if cpacking_raiz and cpacking_raiz in mapa_costing:
                itens = mapa_costing[cpacking_raiz]
                for item in itens:
                    resultado_unificado.append([
                        "GIG",
                        "CBase",
                        "01/07/2026",
                        companhia_detectada,
                        row.get("Servico"),
                        None,                          # Preenchido via propagação
                        "CostAb",
                        item.get("Class_Costing"),
                        item["Codigo"],
                        item["Descricao_Item"],
                        row.get("Classe"),
                        row.get("Rotation"),
                        item.get("YIELD"),
                        item.get("QTNET"),
                        item.get("UN_NOVO"),
                        item.get("CUSTO_FINAL")
                    ])

        colunas = [
            "Base", "Cod Menu Grid", "Base de custo", "Cia", "Serviço", "Cardapio",
            "Relatorio", "Class", "Codigo inflightor", "Produto e receita", "Classe", "Ciclo",
            "Yield", "Quantidade", "Unidade", "Custo Unitario"
        ]

        df_final = pd.DataFrame(resultado_unificado, columns=colunas)

        # Garantir que a coluna ‘Produto e receita’ fique totalmente em maiúsculo
        df_final["Produto e receita"] = df_final["Produto e receita"].astype(str).str.upper()

        # --- PROPAGAÇÃO DO CARDÁPIO DO DATAFRAME FINAL ---
        cardapio_atual_propagado = None

        for i in range(len(df_final)):
            codigo_inflightor = str(df_final.iloc[i, 8]).strip().upper() # Coluna I (índice 8)
            nome_produto_receita = df_final.iloc[i, 9]                   # Coluna J (índice 9)

            if codigo_inflightor.startswith(("T", "C")):
                cardapio_atual_propagado = nome_produto_receita

            df_final.iloc[i, 5] = cardapio_atual_propagado               # Coluna F (índice 5)

        # Converter tipos numéricos para garantir formatação correta
        df_final["Yield"] = pd.to_numeric(df_final["Yield"], errors="coerce")
        df_final["Quantidade"] = pd.to_numeric(df_final["Quantidade"], errors="coerce")
        df_final["Custo Unitario"] = pd.to_numeric(df_final["Custo Unitario"], errors="coerce")

        # Ordenação: Primeiro por Serviço e depois por Ciclo (Rotation)
        df_final["temp_ordenacao_ciclo"] = pd.to_numeric(df_final["Ciclo"], errors="coerce").fillna(9999)
        df_final = df_final.sort_values(by=["Serviço", "temp_ordenacao_ciclo"], ascending=[True, True])
        df_final.drop(columns=["temp_ordenacao_ciclo"], inplace=True, errors="ignore")

        # Formatação e Salvamento do Arquivo Excel usando formatação nativa do openpyxl
        saida_unificada = f"MRP Costing CBase {companhia_detectada}.xlsx"
        with pd.ExcelWriter(saida_unificada, engine="openpyxl") as writer:
            df_final.to_excel(writer, sheet_name="Dados", index=False)

            # Recupera a planilha para aplicar estilos do Excel nas colunas
            workbook  = writer.book
            worksheet = writer.sheets["Dados"]

            # Identifica as colunas de interesse por nome
            col_yield_idx = df_final.columns.get_loc("Yield") + 1
            col_qtd_idx = df_final.columns.get_loc("Quantidade") + 1
            col_custo_idx = df_final.columns.get_loc("Custo Unitario") + 1

            # Formatação de número nativa para Excel
            # Yield: Percentagem (0.00% ou 0%)
            # Quantidade: Número com duas casas decimais (0.00)
            # Custo Unitario: Moeda local com duas casas decimais (R$ #,##0.00)
            for row_idx in range(2, worksheet.max_row + 1):
                # Yield -> formatar como porcentagem
                cell_yield = worksheet.cell(row=row_idx, column=col_yield_idx)
                if cell_yield.value is not None:
                    cell_yield.number_format = "0.00%"

                # Quantidade -> formatar com duas casas decimais
                cell_qtd = worksheet.cell(row=row_idx, column=col_qtd_idx)
                if cell_qtd.value is not None:
                    cell_qtd.number_format = "0.00"

                # Custo Unitario -> formatar como moeda
                cell_custo = worksheet.cell(row=row_idx, column=col_custo_idx)
                if cell_custo.value is not None:
                    cell_custo.number_format = '"R$" #,##0.00'

        print(f"\nSucesso! Planilha unificada gerada no mesmo padrão e salva em: {saida_unificada}")
        files.download(saida_unificada)
    else:
        print("Houve falha ao cruzar dados com o arquivo Costing.")
else:
    print("Houve falha ao carregar os dados de origem.")

--- Selecione o arquivo Meal Definition ---


Saving TAP GIG MEAL DEFINITION_MAR 26.xlsx to TAP GIG MEAL DEFINITION_MAR 26 (6).xlsx
Companhia identificada na célula A1: TP TAP Air Portugal
--- Selecione o arquivo Costing Aberto---


Saving COSTING ABERTO TAP GIG 20.01.26.xls to COSTING ABERTO TAP GIG 20.01.26 (6).xls

Sucesso! Planilha unificada gerada no mesmo padrão e salva em: MRP Costing CBase TP TAP Air Portugal.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import re
import io
from google.colab import files

# --- FUNÇÕES AUXILIARES DE SUPORTE ---

def normalizar_texto(texto):
    """Remove espaços extras, padroniza caracteres especiais e converte para maiúsculo."""
    if pd.isna(texto):
        return ""
    # Mantém apenas caracteres alfanuméricos básicos para evitar quebras por caracteres invisíveis
    return re.sub(r'[^A-Z0-9]', '', str(texto).strip().upper())

def obter_classificacao_classe(codigo, companhia):
    if pd.isna(codigo):
        return None

    codigo_str = str(codigo).strip().upper()
    cia_letras = re.sub(r'[^A-Z]', '', companhia.upper())

    if codigo_str.startswith(('R', 'Y')):
        return "R"
    elif codigo_str.startswith('13'):
        return "P"
    elif codigo_str.startswith(('T', 'C')):
        if len(codigo_str) > 1 and codigo_str[1].isdigit():
            return None
        return "CIA"
    else:
        prefixo_cia = cia_letras[:2] if len(cia_letras) >= 2 else cia_letras
        if prefixo_cia and codigo_str.startswith(prefixo_cia):
            return "CIA"
        return "ESP"

# --- PROCESSAMENTO DOS ARQUIVOS ---

def processar_meal_definition_unificado():
    print("--- Selecione o arquivo Meal Definition ---")
    uploaded = files.upload()
    if not uploaded:
        return pd.DataFrame(), set(), "TAP"

    arquivo = list(uploaded.keys())[0]

    # Identificação dinâmica da Cia na célula A1
    df_identificacao = pd.read_excel(io.BytesIO(uploaded[arquivo]), header=None, nrows=1)
    companhia_detectada = "TAP"
    if not df_identificacao.empty and df_identificacao.shape[1] > 0:
        valor_a1 = str(df_identificacao.iloc[0, 0]).strip()
        if valor_a1 and valor_a1.lower() != 'nan':
            companhia_detectada = valor_a1
            print(f"Companhia identificada na célula A1: {companhia_detectada}")

    # Lê os dados úteis
    df = pd.read_excel(io.BytesIO(uploaded[arquivo]), header=1)

    # Normalização em lote das colunas
    df.columns = df.columns.astype(str).str.strip().str.upper()
    df.columns = df.columns.str.replace(r'[^A-Z0-9]', '', regex=True)

    # Mapeamento inteligente de colunas por índice físico
    colunas_originais = list(df.columns)
    mapa_renomear = {}
    if len(colunas_originais) > 0: mapa_renomear[colunas_originais[0]] = "Servico"
    if len(colunas_originais) > 3: mapa_renomear[colunas_originais[3]] = "Classe"
    if len(colunas_originais) > 6: mapa_renomear[colunas_originais[6]] = "Rotation"
    if len(colunas_originais) > 9: mapa_renomear[colunas_originais[9]] = "Cardapio_Base_ColJ"

    # Renomeação por text aproximado para CPACKINGLIST
    for col in df.columns:
        if "PACK" in col or "CPACK" in col:
            mapa_renomear[col] = "CPACKINGLIST"
            break

    mapa_geral = {"PRODTEXT": "Descricao", "PROD_TEXT": "Descricao", "MODE": "Mode", "VALUE": "Value"}
    mapa_geral.update(mapa_renomear)
    df = df.rename(columns=mapa_geral)

    if "CPACKINGLIST" not in df.columns:
        print(f"\n[ERRO] Não foi possível encontrar a coluna 'CPACKINGLIST'!")
        return pd.DataFrame(), set(), companhia_detectada

    df["Class_Meal"] = df["CPACKINGLIST"].apply(lambda x: obter_classificacao_classe(x, companhia_detectada))

    # Chaves normalizadas para cruzamento à prova de falhas de digitação
    df["CPACKINGLIST_KEY"] = df["CPACKINGLIST"].apply(normalizar_texto)
    set_cpacking_chaves = set(df["CPACKINGLIST_KEY"].dropna().unique())

    colunas_necessarias = ["Servico", "Rotation", "CPACKINGLIST", "CPACKINGLIST_KEY", "Descricao", "Mode", "Value", "Classe", "Cardapio_Base_ColJ", "Class_Meal"]
    cols_existentes = [c for c in colunas_necessarias if c in df.columns]

    return df[cols_existentes].copy(), set_cpacking_chaves, companhia_detectada

def processar_costing_sequencial(set_codigos_meal, companhia):
    print("--- Selecione o arquivo Costing Aberto ---")
    uploaded = files.upload()
    if not uploaded:
        return {}, {}

    arquivo = list(uploaded.keys())[0]
    df = pd.read_excel(io.BytesIO(uploaded[arquivo]), header=None)

    mapa_receitas = {}
    mapa_custos_pais = {}
    cpacking_atual = None

    # Loop de varredura das linhas
    for idx, row in df.iterrows():
        codigo_bruto = row[0]
        if pd.isna(codigo_bruto):
            continue

        # Normaliza o código para bater de forma idêntica com o set de chaves
        codigo_chave = normalizar_texto(codigo_bruto)

        if codigo_chave in set_codigos_meal:
            cpacking_atual = codigo_chave
            mapa_receitas[cpacking_atual] = []

            qtgross_pai = row[6]
            un_val_pai = row[7]
            custo_pai = qtgross_pai if pd.notnull(qtgross_pai) and str(qtgross_pai).strip() != "" else (un_val_pai if pd.notnull(un_val_pai) else 0.0)
            mapa_custos_pais[cpacking_atual] = custo_pai
            continue

        if cpacking_atual is not None:
            classe_item_calculada = obter_classificacao_classe(codigo_bruto, companhia)

            qtgross = row[6]
            un_val = row[7]
            # CORREÇÃO: un_val_idx foi alterado para un_val para evitar NameError
            custo_unitario_final = qtgross if pd.notnull(qtgross) and str(qtgross).strip() != "" else (un_val if pd.notnull(un_val) else 0.0)

            mapa_receitas[cpacking_atual].append({
                "Codigo": str(codigo_bruto).strip(),
                "Descricao_Item": str(row[1]).strip() if pd.notnull(row[1]) else "",
                "QTNET": row[4] if pd.notnull(row[4]) else None,
                "YIELD": row[3] if pd.notnull(row[3]) else None,
                "UN_NOVO": row[5] if pd.notnull(row[5]) else None,
                "CUSTO_FINAL": custo_unitario_final,
                "Class_Costing": classe_item_calculada
            })

    return mapa_receitas, mapa_custos_pais

# --- EXECUÇÃO OTIMIZADA ---

df_meal_unif, set_codigos_meal, companhia_detectada = processar_meal_definition_unificado()

if not df_meal_unif.empty:
    mapa_costing, mapa_custos_pais = processar_costing_sequencial(set_codigos_meal, companhia_detectada)

    if mapa_costing:
        resultado_unificado = []

        for row in df_meal_unif.to_dict("records"):
            cpacking_raiz = row.get("CPACKINGLIST_KEY")
            descricao_original = row.get("Descricao")
            cardapio_col_j = row.get("Cardapio_Base_ColJ")

            nome_produto_pai = cardapio_col_j if pd.notnull(cardapio_col_j) else descricao_original

            resultado_unificado.append([
                "GIG", "CBase", "01/07/2026", companhia_detectada, row.get("Servico"),
                None, "MealD", row.get("Class_Meal"), row.get("CPACKINGLIST"),
                str(nome_produto_pai).upper(), row.get("Classe"), row.get("Rotation"),
                None, None, "UN", mapa_custos_pais.get(cpacking_raiz, 0.0)
            ])

            if cpacking_raiz in mapa_costing:
                for item in mapa_costing[cpacking_raiz]:
                    resultado_unificado.append([
                        "GIG", "CBase", "01/07/2026", companhia_detectada, row.get("Servico"),
                        None, "CostAb", item.get("Class_Costing"), item["Codigo"],
                        item["Descricao_Item"].upper(), row.get("Classe"), row.get("Rotation"),
                        item.get("YIELD"), item.get("QTNET"), item.get("UN_NOVO"), item.get("CUSTO_FINAL")
                    ])

        colunas = [
            "Base", "Cod Menu Grid", "Base de custo", "Cia", "Serviço", "Cardapio",
            "Relatorio", "Class", "Codigo inflightor", "Produto e receita", "Classe", "Ciclo",
            "Yield", "Quantidade", "Unidade", "Custo Unitario"
        ]

        df_final = pd.DataFrame(resultado_unificado, columns=colunas)

        # --- PROPAGAÇÃO EFICIENTE (VETORIZADA) COM PANDAS ---
        # Identifica onde estão as linhas de controle (começam com T ou C) e propaga usando .ffill()
        eh_cabecalho = df_final["Codigo inflightor"].astype(str).str.strip().str.upper().str.startswith(("T", "C"))
        df_final["Cardapio"] = np.where(eh_cabecalho, df_final["Produto e receita"], np.nan)
        df_final["Cardapio"] = df_final["Cardapio"].ffill()

        # Conversões seguras
        for col in ["Yield", "Quantidade", "Custo Unitario"]:
            df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

        # Ordenação
        df_final["temp_ordenacao_ciclo"] = pd.to_numeric(df_final["Ciclo"], errors="coerce").fillna(9999)
        df_final = df_final.sort_values(by=["Serviço", "temp_ordenacao_ciclo"], ascending=[True, True])
        df_final.drop(columns=["temp_ordenacao_ciclo"], inplace=True, errors="ignore")

        # Geração do Excel formatado
        saida_unificada = f"MRP Costing CBase {companhia_detectada}.xlsx"
        with pd.ExcelWriter(saida_unificada, engine="openpyxl") as writer:
            df_final.to_excel(writer, sheet_name="Dados", index=False)

            workbook  = writer.book
            worksheet = writer.sheets["Dados"]

            # Aplica formatações numéricas rapidamente no Excel
            col_yield_idx = df_final.columns.get_loc("Yield") + 1
            col_qtd_idx = df_final.columns.get_loc("Quantidade") + 1
            col_custo_idx = df_final.columns.get_loc("Custo Unitario") + 1

            for row_idx in range(2, worksheet.max_row + 1):
                cell_yield = worksheet.cell(row=row_idx, column=col_yield_idx)
                if cell_yield.value is not None:
                    cell_yield.number_format = "0.00%"

                cell_qtd = worksheet.cell(row=row_idx, column=col_qtd_idx)
                if cell_qtd.value is not None:
                    cell_qtd.number_format = "0.00"

                cell_custo = worksheet.cell(row=row_idx, column=col_custo_idx)
                if cell_custo.value is not None:
                    cell_custo.number_format = '"R$" #,##0.00'

        print(f"\nSucesso! Planilha unificada gerada com código otimizado em: {saida_unificada}")
        files.download(saida_unificada)
    else:
        print("Houve falha ao cruzar dados com o arquivo Costing.")
else:
    print("Houve falha ao carregar os dados de origem.")

--- Selecione o arquivo Meal Definition ---


Saving TAP GIG MEAL DEFINITION_MAR 26.xlsx to TAP GIG MEAL DEFINITION_MAR 26 (3).xlsx
Companhia identificada na célula A1: TP TAP Air Portugal
--- Selecione o arquivo Costing Aberto ---


Saving COSTING ABERTO TAP GIG 20.01.26.xls to COSTING ABERTO TAP GIG 20.01.26 (3).xls

Sucesso! Planilha unificada gerada com código otimizado em: MRP Costing CBase TP TAP Air Portugal.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>